In [2]:
import pandas as pd
import numpy as np
import re
from datasets import Dataset
import ast
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
Label_data=pd.read_excel("train_fixed.xlsx")
unlabeled_data=pd.read_excel("unlabeled_fixed.xlsx")
validation_data=pd.read_excel("DeepX_validation.xlsx")

In [4]:
Label_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1971 entries, 0 to 1970
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   review_id          1971 non-null   int64 
 1   review_text        1971 non-null   object
 2   star_rating        1971 non-null   int64 
 3   date               1971 non-null   object
 4   business_name      1971 non-null   object
 5   business_category  1971 non-null   object
 6   platform           1971 non-null   object
 7   aspects            1971 non-null   object
 8   aspect_sentiments  1971 non-null   object
dtypes: int64(2), object(7)
memory usage: 138.7+ KB


In [5]:
Label_data.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


In [6]:
unlabeled_data.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform
0,1,Incroyablement grand avec des belles boutiques...,5,قبل 7 ساعات,مول سيتي ستارز.,مركز تسوق,google_maps
1,2,زحمه جدا,5,قبل 12 ساعة,مول سيتي ستارز.,مركز تسوق,google_maps
2,3,حلو فخم كشخة محترم ورايق ينفع للعوائل الخليجي...,5,قبل يوم واحد,مول سيتي ستارز.,مركز تسوق,google_maps
3,4,طبعا غني عن التعريف بتاع البشوات,5,قبل يوم واحد,مول سيتي ستارز.,مركز تسوق,google_maps
4,5,Centro commerciale al Cairo... Molto grande e ...,5,قبل يومين (2),مول سيتي ستارز.,مركز تسوق,google_maps


In [7]:
print(Label_data["business_category"].unique())

['ecommerce' 'كافيه' 'مطعم' 'مطعم مأكولات ومشروبات' 'مطعم يمني'
 'entertainment' 'مطعم أطباق اللحوم' 'مقهى' 'عِيادة أسنان' 'صيدلية'
 'مركز طبي' 'travel' 'transport' 'مطعم مأكولات الشرق الأوسط' 'مطعم بيتزا'
 'مطعم فلافل' 'مطعم مأكولات مشوية' 'مطعم مأكولات لبنانية'
 'مطعم مأكولات سورية' 'فندق' 'متجر ملابس حريمي' 'food_delivery'
 'مستشفى متخصص' 'صالون تجميل' 'real_estate' 'مطعم مأكولات تركية'
 'مطعم دجاج' 'مطعم مأكولات إيطالية' 'طبيب أسنان' 'صالة رياضة'
 'مطعم مأكولات أمريكية' 'مستشفى' 'مطعم شاورما' 'متجر بقالة'
 'مطعم مأكولات هندية' 'مركز تسوق' 'عيادة طبية' 'متجر ملابس'
 'مطعم وجبات سريعة' 'سوبرماركت' 'مطعم للإفطار' 'مطعم ببوفيه'
 'مطعم مأكولات مصرية' 'متجر ملابس رجالي' 'مستشفى جامعي' 'مصفف الشعر'
 'المستشفى العسكري' 'مطعم عائلي' 'عيادة الخصوبة' 'غرفة لياقة'
 'عيادة متخصصة' 'متجر ملابس رياضية' 'مطعم مأكولات فرنسية'
 'صالون عناية بالأظافر' 'سوق' 'مطعم مأكولات بحرية' 'صالون حلاقة'
 'مطعم كريب' 'متجر' 'عيادة جراحة التجميل' 'مطعم مأكولات من جنوب إفريقيا'
 'مستشفى عام' 'المستشفى الحكومي' 'منف

In [8]:
Label_data["platform"].unique()

array(['play_store', 'google_maps'], dtype=object)

In [9]:
Label_data["aspects"] = Label_data["aspects"].apply(ast.literal_eval)

unique_aspects = set()

for aspects in Label_data["aspects"]:
    unique_aspects.update(aspects)

print(unique_aspects)



{'app_experience', 'delivery', 'service', 'price', 'general', 'none', 'food', 'cleanliness', 'ambiance'}


In [10]:
def clean_arabic(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # normalize Arabic letters
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)

    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

Label_data["review_text"] = Label_data["review_text"].apply(clean_arabic)

In [11]:
from datetime import datetime
from datetime import timedelta


reference_date=pd.Timestamp.today()

def tokenize(text):
    return text.split()

# time vocab (days)
time_units = {
    "يوم": 1,
    "يومين": 2,
    "ايام": 1,
    
    "شهر": 30,
    "اشهر": 30,
    
    "سنه": 365,
    "سنين": 365,
    "سنتين": 2 * 365,
    "سنوات": 365
}

# extract number
def extract_number(tokens):
    for t in tokens:
        if t.isdigit():
            return int(t)
    return None

# convert to DATETIME
def parse_to_datetime(text):
    text = clean_arabic(text)

    # absolute date ----
    try:
        return pd.to_datetime(text)
    except:
        pass

    # relative Arabic ----
    tokens = tokenize(text)
    number = extract_number(tokens)
    unit_value = None

    for t in tokens:
        if t in time_units:
            unit_value = time_units[t]
            break

    if unit_value is not None:
        if number is None:
            number = 1
        
        days = number * unit_value
        return reference_date - timedelta(days=days)

    return pd.NaT

# apply
Label_data['date_parsed'] = Label_data['date'].apply(parse_to_datetime)

# extract features
Label_data['year'] = Label_data['date_parsed'].dt.year
Label_data['month'] = Label_data['date_parsed'].dt.month
Label_data['day_of_week'] = Label_data['date_parsed'].dt.dayofweek  # 0=Mon , 6=Sunday and so on

# handle missing
Label_data['year'] = Label_data['year'].fillna(Label_data['year'].median())
Label_data['month'] = Label_data['month'].fillna(Label_data['month'].median())
Label_data['day_of_week'] = Label_data['day_of_week'].fillna(Label_data['day_of_week'].median())
Label_data['date_parsed']

Label_data=Label_data.drop(columns=["business_name","review_id","date"])

In [12]:
Label_data["aspect_sentiments"].apply(type).value_counts()

aspect_sentiments
<class 'str'>    1971
Name: count, dtype: int64

In [13]:
import ast
Label_data["aspect_sentiments"] = Label_data["aspect_sentiments"].apply(ast.literal_eval)

In [14]:

Label_data['business_category'] = Label_data['business_category'].astype(str)
Label_data['business_category'] = Label_data['business_category'].apply(
    lambda x: x.strip().lower()
)
Label_data['business_category'] = Label_data['business_category'].str.replace(r'\s+', ' ', regex=True)
print(Label_data['business_category'].unique())
category_mapping = {
    # Restaurants (all مطعم variations)
    "مطعم": "restaurant",
    "مطعم مأكولات ومشروبات": "restaurant",
    "مطعم يمني": "restaurant",
    "مطعم أطباق اللحوم": "restaurant",
    "مطعم مأكولات الشرق الأوسط": "restaurant",
    "مطعم بيتزا": "restaurant",
    "مطعم فلافل": "restaurant",
    "مطعم مأكولات مشوية": "restaurant",
    "مطعم مأكولات لبنانية": "restaurant",
    "مطعم مأكولات سورية": "restaurant",
    "مطعم مأكولات تركية": "restaurant",
    "مطعم دجاج": "restaurant",
    "مطعم مأكولات إيطالية": "restaurant",
    "مطعم مأكولات أمريكية": "restaurant",
    "مطعم شاورما": "restaurant",
    "مطعم مأكولات هندية": "restaurant",
    "مطعم وجبات سريعة": "restaurant",
    "مطعم للإفطار": "restaurant",
    "مطعم ببوفيه": "restaurant",
    "مطعم مأكولات مصرية": "restaurant",
    "مطعم عائلي": "restaurant",
    "مطعم مأكولات فرنسية": "restaurant",
    "مطعم مأكولات بحرية": "restaurant",
    "مطعم كريب": "restaurant",
    "مطعم مأكولات من جنوب إفريقيا": "restaurant",
    # Cafes
    "كافيه": "cafe",
    "مقهى": "cafe",
    # Medical (hospitals, clinics, pharmacies, dentists)
    "عِيادة أسنان": "medical",
    "صيدلية": "medical",
    "مركز طبي": "medical",
    "مستشفى متخصص": "medical",
    "طبيب أسنان": "medical",
    "مستشفى": "medical",
    "عيادة طبية": "medical",
    "مستشفى جامعي": "medical",
    "المستشفى العسكري": "medical",
    "عيادة الخصوبة": "medical",
    "عيادة متخصصة": "medical",
    "مستشفى عام": "medical",
    "المستشفى الحكومي": "medical",
    "مستشفى خاصة": "medical",
    "عيادة جراحة التجميل": "medical",
    # Gyms and fitness rooms
    "صالة رياضة": "gym",
    "غرفة لياقة": "gym",
    "برنامج اللياقة البدنية": "fitness_program",  # or keep as "gym"
    # Beauty salons (including nail salon)
    "صالون تجميل": "salon",
    "صالون عناية بالأظافر": "salon",
    # Barbers / hairdressers
    "مصفف الشعر": "barber",
    "صالون حلاقة": "barber",
    # Hotel
    "فندق": "hotel",
    # Retail (general stores, markets, malls, outlets)
    "متجر": "retail",
    "منفذ بيع بالتجزئة": "retail",
    "سوق": "retail",
    "مركز تسوق": "retail",   # mall
    "متجر بقالة": "supermarket",
    "سوبرماركت": "supermarket",
    "متجر بقالة راقٍ": "supermarket",
    # Clothing stores
    "متجر ملابس حريمي": "clothing_store",
    "متجر ملابس": "clothing_store",
    "متجر ملابس رجالي": "clothing_store",
    "متجر ملابس رياضية": "clothing_store",
    # Other categories
    "ecommerce": "ecommerce",
    "travel": "travel",
    "transport": "transport",
    "food_delivery": "food_delivery",
    "real_estate": "real_estate",
    "entertainment": "entertainment",
}
Label_data['business_domain'] = Label_data['business_category'].map(category_mapping)
print(Label_data['business_domain'].value_counts())
Label_data = Label_data.drop('business_category', axis = 1)
Label_data['text_with_domain'] = "[domain: " + Label_data['business_domain'] + "] " + Label_data['review_text']
Label_data = Label_data.drop('business_domain', axis = 1)
Label_data = Label_data.drop('review_text', axis = 1)

['ecommerce' 'كافيه' 'مطعم' 'مطعم مأكولات ومشروبات' 'مطعم يمني'
 'entertainment' 'مطعم أطباق اللحوم' 'مقهى' 'عِيادة أسنان' 'صيدلية'
 'مركز طبي' 'travel' 'transport' 'مطعم مأكولات الشرق الأوسط' 'مطعم بيتزا'
 'مطعم فلافل' 'مطعم مأكولات مشوية' 'مطعم مأكولات لبنانية'
 'مطعم مأكولات سورية' 'فندق' 'متجر ملابس حريمي' 'food_delivery'
 'مستشفى متخصص' 'صالون تجميل' 'real_estate' 'مطعم مأكولات تركية'
 'مطعم دجاج' 'مطعم مأكولات إيطالية' 'طبيب أسنان' 'صالة رياضة'
 'مطعم مأكولات أمريكية' 'مستشفى' 'مطعم شاورما' 'متجر بقالة'
 'مطعم مأكولات هندية' 'مركز تسوق' 'عيادة طبية' 'متجر ملابس'
 'مطعم وجبات سريعة' 'سوبرماركت' 'مطعم للإفطار' 'مطعم ببوفيه'
 'مطعم مأكولات مصرية' 'متجر ملابس رجالي' 'مستشفى جامعي' 'مصفف الشعر'
 'المستشفى العسكري' 'مطعم عائلي' 'عيادة الخصوبة' 'غرفة لياقة'
 'عيادة متخصصة' 'متجر ملابس رياضية' 'مطعم مأكولات فرنسية'
 'صالون عناية بالأظافر' 'سوق' 'مطعم مأكولات بحرية' 'صالون حلاقة'
 'مطعم كريب' 'متجر' 'عيادة جراحة التجميل' 'مطعم مأكولات من جنوب إفريقيا'
 'مستشفى عام' 'المستشفى الحكومي' 'منف

In [15]:
Label_data.head()

,star_rating,platform,aspects,aspect_sentiments,date_parsed,year,month,day_of_week,text_with_domain
0,3,play_store,"[app_experience, delivery]","{'app_experience': 'negative', 'delivery': 'ne...",2026-03-08 00:00:00.000000,2026.0,3.0,6.0,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...
1,5,google_maps,"[cleanliness, ambiance, service]","{'cleanliness': 'positive', 'ambiance': 'posit...",2026-04-20 14:45:37.370151,2026.0,4.0,0.0,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...
2,1,google_maps,"[service, delivery, food]","{'service': 'negative', 'delivery': 'negative'...",2026-03-25 14:45:37.370151,2026.0,3.0,2.0,[domain: restaurant] تجربه سيئه سالتهم الاكل ه...
3,5,google_maps,[general],{'general': 'positive'},2026-03-25 14:45:37.370151,2026.0,3.0,2.0,[domain: restaurant] احلي مكان فزايد
4,4,google_maps,"[food, price]","{'food': 'positive', 'price': 'positive'}",2025-04-24 14:45:37.370151,2025.0,4.0,3.0,[domain: restaurant] الفطير حلو جدا الاحجام تح...


In [16]:
unique_aspects = set()

for aspects in Label_data["aspects"]:
    unique_aspects.update(aspects)

print(unique_aspects)

{'app_experience', 'delivery', 'service', 'price', 'general', 'none', 'food', 'cleanliness', 'ambiance'}


In [17]:
Label_data["aspects"] = Label_data["aspects"].apply(
    lambda x: [a.lower().strip() for a in x]
)
Label_data.head()

,star_rating,platform,aspects,aspect_sentiments,date_parsed,year,month,day_of_week,text_with_domain
0,3,play_store,"[app_experience, delivery]","{'app_experience': 'negative', 'delivery': 'ne...",2026-03-08 00:00:00.000000,2026.0,3.0,6.0,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...
1,5,google_maps,"[cleanliness, ambiance, service]","{'cleanliness': 'positive', 'ambiance': 'posit...",2026-04-20 14:45:37.370151,2026.0,4.0,0.0,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...
2,1,google_maps,"[service, delivery, food]","{'service': 'negative', 'delivery': 'negative'...",2026-03-25 14:45:37.370151,2026.0,3.0,2.0,[domain: restaurant] تجربه سيئه سالتهم الاكل ه...
3,5,google_maps,[general],{'general': 'positive'},2026-03-25 14:45:37.370151,2026.0,3.0,2.0,[domain: restaurant] احلي مكان فزايد
4,4,google_maps,"[food, price]","{'food': 'positive', 'price': 'positive'}",2025-04-24 14:45:37.370151,2025.0,4.0,3.0,[domain: restaurant] الفطير حلو جدا الاحجام تح...


In [18]:
rows = []

for _, row in Label_data.iterrows():
    review = row["text_with_domain"]
    aspect_dict = row["aspect_sentiments"]
    date_parsed = row["date_parsed"]
    palt_form= row["platform"]

    for aspect, sentiment in aspect_dict.items():
        rows.append({
            "text": review,
            "aspect": aspect,
                "year": row["year"],
                "month": row["month"],
                "day_of_week": row["day_of_week"],
                "platform": palt_form,
                "label": sentiment

            
        })

final_df = pd.DataFrame(rows)

In [19]:
final_df.head(10)

,text,aspect,year,month,day_of_week,platform,label
0,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...,app_experience,2026.0,3.0,6.0,play_store,negative
1,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...,delivery,2026.0,3.0,6.0,play_store,negative
2,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,cleanliness,2026.0,4.0,0.0,google_maps,positive
3,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,ambiance,2026.0,4.0,0.0,google_maps,positive
4,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,service,2026.0,4.0,0.0,google_maps,positive
5,[domain: restaurant] تجربه سيئه سالتهم الاكل ه...,service,2026.0,3.0,2.0,google_maps,negative
6,[domain: restaurant] تجربه سيئه سالتهم الاكل ه...,delivery,2026.0,3.0,2.0,google_maps,negative
7,[domain: restaurant] تجربه سيئه سالتهم الاكل ه...,food,2026.0,3.0,2.0,google_maps,neutral
8,[domain: restaurant] احلي مكان فزايد,general,2026.0,3.0,2.0,google_maps,positive
9,[domain: restaurant] الفطير حلو جدا الاحجام تح...,food,2025.0,4.0,3.0,google_maps,positive


In [20]:
Label_data=final_df
Label_data.head()


,text,aspect,year,month,day_of_week,platform,label
0,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...,app_experience,2026.0,3.0,6.0,play_store,negative
1,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...,delivery,2026.0,3.0,6.0,play_store,negative
2,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,cleanliness,2026.0,4.0,0.0,google_maps,positive
3,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,ambiance,2026.0,4.0,0.0,google_maps,positive
4,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,service,2026.0,4.0,0.0,google_maps,positive


In [21]:
le = LabelEncoder()
Label_data["label_id"] = le.fit_transform(Label_data["label"])
Label_data=Label_data.drop(columns=["label"])



In [22]:
Label_data.head()


,text,aspect,year,month,day_of_week,platform,label_id
0,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...,app_experience,2026.0,3.0,6.0,play_store,0
1,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...,delivery,2026.0,3.0,6.0,play_store,0
2,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,cleanliness,2026.0,4.0,0.0,google_maps,2
3,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,ambiance,2026.0,4.0,0.0,google_maps,2
4,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,service,2026.0,4.0,0.0,google_maps,2


In [23]:
Label_data.head()

,text,aspect,year,month,day_of_week,platform,label_id
0,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...,app_experience,2026.0,3.0,6.0,play_store,0
1,[domain: ecommerce] لا يوجد الدفع بالبطاقه عند...,delivery,2026.0,3.0,6.0,play_store,0
2,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,cleanliness,2026.0,4.0,0.0,google_maps,2
3,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,ambiance,2026.0,4.0,0.0,google_maps,2
4,[domain: cafe] المكان نضيف وجميل وقعدته تحفه و...,service,2026.0,4.0,0.0,google_maps,2


In [24]:
import pandas as pd
import numpy as np
import torch
import ast

from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# =========================================
# MODELS
# =========================================
models = [
    "aubmindlab/bert-base-arabertv2",
    "UBC-NLP/MARBERT"
]

NUM_LABELS = Label_data["label_id"].nunique()
print("num labels:", NUM_LABELS)


# =========================================
# FINAL TEXT FOR TRAIN
# =========================================
Label_data["final_text"] = (
    Label_data["text"].astype(str)
    + " [ASPECT] " + Label_data["aspect"].astype(str)
    + " [PLATFORM] " + Label_data["platform"].astype(str)
    + " [DATE] year_" + Label_data["year"].astype(str)
    + "_month_" + Label_data["month"].astype(str)
)

print(Label_data[["final_text"]].head())


# =========================================
# FIX UNLABELED DATA
# =========================================
unlabeled_data["review_text"] = unlabeled_data["review_text"].apply(clean_arabic)

unlabeled_data["business_category"] = (
    unlabeled_data["business_category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

unlabeled_data["business_domain"] = (
    unlabeled_data["business_category"]
    .map(category_mapping)
    .fillna("unknown")
)

unlabeled_data["text_with_domain"] = (
    "[domain: "
    + unlabeled_data["business_domain"]
    + "] "
    + unlabeled_data["review_text"]
)


# all aspects from train
all_aspects = Label_data["aspect"].unique().tolist()

print("All aspects:", all_aspects)


# expand unlabeled reviews across all aspects
rows = []

for _, row in unlabeled_data.iterrows():

    for aspect in all_aspects:

        rows.append({
            "review_id": row["review_id"],
            "text": row["text_with_domain"],
            "aspect": aspect,
            "platform": row["platform"]
        })

unlabeled_final = pd.DataFrame(rows)

print("Expanded unlabeled shape:", unlabeled_final.shape)


# final text for unlabeled
unlabeled_final["final_text"] = (
    unlabeled_final["text"].astype(str)
    + " [ASPECT] "
    + unlabeled_final["aspect"].astype(str)
    + " [PLATFORM] "
    + unlabeled_final["platform"].astype(str)
)

print(unlabeled_final.head())


# =========================================
# CROSS VALIDATION
# =========================================
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros((len(Label_data), NUM_LABELS))
test_preds_models = []


# =========================================
# TRAIN LOOP
# =========================================
for model_name in models:

    print("="*50)
    print("TRAINING:", model_name)
    print("="*50)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    collator = DataCollatorWithPadding(tokenizer)

    def tokenize(batch):
        return tokenizer(
            batch["final_text"],
            truncation=True,
            padding="max_length",
            max_length=256
        )

    train_ds = Dataset.from_pandas(
        Label_data[["final_text", "label_id"]]
    )

    unlabeled_ds = Dataset.from_pandas(
        unlabeled_final[["final_text"]]
    )

    train_ds = train_ds.map(tokenize, batched=True)
    unlabeled_ds = unlabeled_ds.map(tokenize, batched=True)

    model_test_preds = np.zeros(
        (len(unlabeled_final), NUM_LABELS)
    )

    for fold, (tr_idx, val_idx) in enumerate(
        skf.split(Label_data, Label_data["label_id"])
    ):

        print(f"Fold {fold}")

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=NUM_LABELS
        )

        args = TrainingArguments(
            output_dir=f"./{model_name.split('/')[-1]}_fold_{fold}",
            learning_rate=2e-5,
            num_train_epochs=3,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            save_strategy="no",
            fp16=torch.cuda.is_available(),
            report_to="none"
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds.select(tr_idx),
            eval_dataset=train_ds.select(val_idx),
            data_collator=collator
        )

        trainer.train()

        # validation predictions
        val_logits = trainer.predict(
            train_ds.select(val_idx)
        ).predictions

        val_probs = torch.softmax(
            torch.tensor(val_logits),
            dim=1
        ).numpy()

        oof_preds[val_idx] += val_probs

        # unlabeled predictions
        test_logits = trainer.predict(
            unlabeled_ds
        ).predictions

        test_probs = torch.softmax(
            torch.tensor(test_logits),
            dim=1
        ).numpy()

        model_test_preds += test_probs / 5

    test_preds_models.append(model_test_preds)


# =========================================
# OOF SCORE
# =========================================
oof_labels = oof_preds.argmax(axis=1)

score = f1_score(
    Label_data["label_id"],
    oof_labels,
    average="macro"
)

print("OOF F1:", score)


# =========================================
# ENSEMBLE
# =========================================
final_unlabeled_preds = np.mean(
    test_preds_models,
    axis=0
)


# =========================================
# PSEUDO LABELING
# =========================================
pseudo_conf = final_unlabeled_preds.max(axis=1)
pseudo_labels = final_unlabeled_preds.argmax(axis=1)

mask = pseudo_conf > 0.97

pseudo_df = unlabeled_final[mask].copy()
pseudo_df["label_id"] = pseudo_labels[mask]

print("Pseudo samples:", len(pseudo_df))


# =========================================
# FINAL TRAIN SET
# =========================================
final_train = pd.concat([
    Label_data[["final_text", "label_id"]],
    pseudo_df[["final_text", "label_id"]]
])

print(final_train.shape)


# =========================================
# SAVE
# =========================================
final_train.to_csv(
    "final_training_data.csv",
    index=False
)

print("DONE ✅")

ModuleNotFoundError: No module named 'torch'

In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers datasets accelerate scikit-learn pandas numpy openpyxl